# DHCF vs DOSAGE-DHCF on MovieLens-100K

End-to-end Kaggle notebook. Same model (DHCF, KDD 2020), same data, same eval — only the **hyperedge construction** differs.

**Why DHCF (not HCCF):** DHCF builds hyperedges with a simple, *fixed*, data-derived rule:
- **User-vertex hypergraph:** one hyperedge per item = the set of users who interacted with it.
- **Item-vertex hypergraph:** one hyperedge per user = the set of items they interacted with.

That's a swappable construction. HCCF's 'hypergraph' was learned end-to-end as a low-rank attention pattern, so swapping it with DOSAGE wasn't a fair comparison. Here it is.

**Two variants:**
1. **DHCF (original):** k-order item-derived / user-derived hyperedges (each item → 1 hyperedge of its users; each user → 1 hyperedge of their items).
2. **DOSAGE-DHCF:** hyperedges from `DOSAGE_TWO_PHASE` (verbatim from your `hgnn-recommendation-system.ipynb`) on the user-user and item-item co-occurrence graphs.

**Eval:** full-ranking Recall@20 / NDCG@20.

DHCF model code is ported verbatim from `dhg.models.hypergraphs.dhcf` (DHG library, the only public DHCF implementation).

In [1]:
!pip install -q networkx scipy tqdm

In [2]:
import os, math, time, zipfile, urllib.request, random
from collections import defaultdict, Counter
import numpy as np
import scipy.sparse as sp
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


## 1. Download MovieLens-100K and build train/test split

Implicit feedback: rating ≥ 4 is a positive interaction. 10-core filter. Per-user 80/20 random split. Same prep used throughout the project.

In [3]:
DATA_DIR = "/kaggle/working/ml-100k"
if not os.path.exists(DATA_DIR):
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-100k.zip",
                                "/kaggle/working/ml-100k.zip")
    with zipfile.ZipFile("/kaggle/working/ml-100k.zip") as z:
        z.extractall("/kaggle/working/")

rows = []
with open(os.path.join(DATA_DIR, "u.data"), "r", encoding="latin-1") as f:
    for line in f:
        u, i, r, t = line.strip().split("	")
        if int(r) >= 4: rows.append((int(u), int(i)))
u_cnt, i_cnt = Counter(u for u,_ in rows), Counter(i for _,i in rows)
rows = [(u,i) for u,i in rows if u_cnt[u]>=10 and i_cnt[i]>=10]
u_ids = sorted({u for u,_ in rows}); i_ids = sorted({i for _,i in rows})
u_map = {u:k for k,u in enumerate(u_ids)}; i_map = {i:k for k,i in enumerate(i_ids)}
rows = [(u_map[u], i_map[i]) for u,i in rows]
n_users, n_items = len(u_ids), len(i_ids)
print(f"Users={n_users}  Items={n_items}  Interactions={len(rows)}")

by_user = defaultdict(list)
for u,i in rows: by_user[u].append(i)
train_pairs, test_by_user = [], {}
rng = random.Random(SEED)
for u, items in by_user.items():
    rng.shuffle(items)
    n_test = max(1, int(0.2*len(items)))
    test_by_user[u] = set(items[:n_test])
    for i in items[n_test:]: train_pairs.append((u,i))
train_user_pos = defaultdict(set)
for u,i in train_pairs: train_user_pos[u].add(i)
print(f"Train: {len(train_pairs)}   Test users: {len(test_by_user)}")


Users=897  Items=824  Interactions=52866
Train: 42644   Test users: 897


## 2. Build the original DHCF hypergraphs

From the DHCF paper / DHG library:
- **`H_ui`** (user-vertex hypergraph): each **item** is one hyperedge containing the users who interacted with it. Shape `(n_users × n_items)`.
- **`H_iu`** (item-vertex hypergraph): each **user** is one hyperedge containing the items they interacted with. Shape `(n_items × n_users)`.

These are *literally* the rows/columns of the binary interaction matrix. No learning, no parameters — just the data.

In [4]:
def build_dhcf_hypergraphs(train_pairs, n_users, n_items):
    """
    Returns:
        H_ui: dense torch tensor (n_users, n_items) -- user-vertex hypergraph
              column j contains 1's for users who interacted with item j (item j IS hyperedge j).
        H_iu: dense torch tensor (n_items, n_users) -- item-vertex hypergraph
              column k contains 1's for items that user k interacted with (user k IS hyperedge k).
    Both column-normalized by sqrt(degree) so H @ H.T is bounded.
    """
    rows_u, cols_i = zip(*train_pairs)
    rows_u = np.array(rows_u); cols_i = np.array(cols_i)
    # H_ui: rows=users, cols=items (each column = one hyperedge = one item)
    R = sp.coo_matrix((np.ones(len(train_pairs), dtype=np.float32), (rows_u, cols_i)),
                       shape=(n_users, n_items)).tocsr()
    # H_iu = R.T
    return R, R.T.tocsr()

H_ui_orig_sp, H_iu_orig_sp = build_dhcf_hypergraphs(train_pairs, n_users, n_items)
print(f'H_ui (user-vertex): shape={H_ui_orig_sp.shape}, nnz={H_ui_orig_sp.nnz}')
print(f'H_iu (item-vertex): shape={H_iu_orig_sp.shape}, nnz={H_iu_orig_sp.nnz}')
print(f'  → DHCF-original gives {n_items} user-hyperedges and {n_users} item-hyperedges')

H_ui (user-vertex): shape=(897, 824), nnz=42644
H_iu (item-vertex): shape=(824, 897), nnz=42644
  → DHCF-original gives 824 user-hyperedges and 897 item-hyperedges


## 3. Build the DOSAGE hypergraphs

Same `DOSAGE_TWO_PHASE` from `hgnn-recommendation-system.ipynb`. We run it on the user-user and item-item co-occurrence graphs to get groups of frequently co-watching users / co-watched items, which become hyperedges.

In [5]:
MIN_SHARED_USER = 5  # ML-100K is smaller; start with 5 (recsys used 15)
MIN_SHARED_ITEM = 5
KNN_AUGMENT_K = 10  # for any node still isolated after threshold, attach edges to its top-K co-occurring neighbors

# ---- User-user co-occurrence graph ----
item_users = defaultdict(set)
for u,i in train_pairs: item_users[i].add(u)
shared_uu = defaultdict(int)
for item, users in tqdm(item_users.items(), desc="count uu shared"):
    users = list(users)
    for i in range(len(users)):
        for j in range(i+1, len(users)):
            shared_uu[(min(users[i],users[j]), max(users[i],users[j]))] += 1

# Per-user neighbor lists (with counts) — needed for KNN-augment and KNN-attachment fallback
uu_nbrs = defaultdict(list)  # user -> list of (other_user, count)
for (a,b), c in shared_uu.items():
    uu_nbrs[a].append((b, c)); uu_nbrs[b].append((a, c))

G_user = nx.Graph(); G_user.add_nodes_from(range(n_users))
for (a,b), c in shared_uu.items():
    if c >= MIN_SHARED_USER: G_user.add_edge(a, b)

# A1: KNN-augment — guarantee every user has at least KNN_AUGMENT_K neighbors
iso_before = sum(1 for n,d in G_user.degree() if d == 0)
augmented = 0
for u in range(n_users):
    if G_user.degree(u) >= KNN_AUGMENT_K: continue
    # pick top-K by count (regardless of threshold)
    top = sorted(uu_nbrs[u], key=lambda x: -x[1])[:KNN_AUGMENT_K]
    for v, _ in top:
        if not G_user.has_edge(u, v):
            G_user.add_edge(u, v); augmented += 1
iso_u = sum(1 for n,d in G_user.degree() if d == 0)
print(f"G_user: {G_user.number_of_nodes()} nodes, {G_user.number_of_edges()} edges, "
      f"isolated_before={iso_before} isolated_after_KNN={iso_u}, KNN_edges_added={augmented}")

# ---- Item-item co-occurrence graph ----
user_items = defaultdict(set)
for u,i in train_pairs: user_items[u].add(i)
shared_ii = defaultdict(int)
for u, items in tqdm(user_items.items(), desc="count ii shared"):
    items = list(items)
    for i in range(len(items)):
        for j in range(i+1, len(items)):
            shared_ii[(min(items[i],items[j]), max(items[i],items[j]))] += 1

ii_nbrs = defaultdict(list)
for (a,b), c in shared_ii.items():
    ii_nbrs[a].append((b, c)); ii_nbrs[b].append((a, c))

G_item = nx.Graph(); G_item.add_nodes_from(range(n_items))
for (a,b), c in shared_ii.items():
    if c >= MIN_SHARED_ITEM: G_item.add_edge(a, b)

iso_before = sum(1 for n,d in G_item.degree() if d == 0)
augmented = 0
for u in range(n_items):
    if G_item.degree(u) >= KNN_AUGMENT_K: continue
    top = sorted(ii_nbrs[u], key=lambda x: -x[1])[:KNN_AUGMENT_K]
    for v, _ in top:
        if not G_item.has_edge(u, v):
            G_item.add_edge(u, v); augmented += 1
iso_i = sum(1 for n,d in G_item.degree() if d == 0)
print(f"G_item: {G_item.number_of_nodes()} nodes, {G_item.number_of_edges()} edges, "
      f"isolated_before={iso_before} isolated_after_KNN={iso_i}, KNN_edges_added={augmented}")


count uu shared: 100%|██████████| 824/824 [00:01<00:00, 476.88it/s]


G_user: 897 nodes, 145653 edges, isolated_before=3 isolated_after_KNN=0, KNN_edges_added=145


count ii shared: 100%|██████████| 897/897 [00:01<00:00, 682.27it/s]


G_item: 824 nodes, 108771 edges, isolated_before=2 isolated_after_KNN=0, KNN_edges_added=185


In [6]:
# DOSAGE primitives — verbatim from hgnn-recommendation-system.ipynb
stats = {'subsets_checked':0,'density_calls':0,'distance_calls':0,'objective_calls':0}

def DENSITY(G):
    if G.number_of_nodes() == 0: return 0
    return G.number_of_edges() / G.number_of_nodes()

def DISTANCE(G1, G2):
    U, Z = set(G1.nodes()), set(G2.nodes())
    if not U or not Z: return 2
    if U == Z: return 0
    return 2 - (len(U & Z) ** 2) / (len(U) * len(Z))

def OBJECTIVE(W, lam):
    td = sum(DENSITY(g) for g in W)
    tdist = sum(DISTANCE(W[i], W[j]) for i in range(len(W)) for j in range(i+1, len(W)))
    return td + lam * tdist

def DENSESTSUBGRAPH(G, alpha, beta):
    G_current = G.copy(); G_best = None; best_density = 0
    while G_current.number_of_nodes() > 0:
        n = G_current.number_of_nodes()
        if n < alpha: break
        if alpha <= n <= beta:
            d = DENSITY(G_current)
            if d > best_density:
                best_density = d; G_best = G_current.copy()
        degrees = dict(G_current.degree())
        min_deg = min(degrees.values())
        G_current.remove_nodes_from([v for v,d in degrees.items() if d == min_deg])
    return G_best

In [7]:
def DOSAGE_TWO_PHASE(G, k, lam, alpha, beta, trials=100, phase2_extra=30,
                     local_cap=80, cov_sample_size=15, verbose=True, nbr_lookup=None):
    """Verbatim port of recsys notebook's DOSAGE_TWO_PHASE + verbose per-hyperedge printing
    + KNN-attachment fallback (replaces singleton fallback when nbr_lookup is provided)."""
    global stats
    W = []
    membership = {n: 0 for n in G.nodes()}  # how many hyperedges each node is in
    N_total = G.number_of_nodes()
    t_overall = time.time()
    if verbose:
        print("=" * 60)
        print("PHASE 1: Coverage-first")
        print("=" * 60)
    while len(W) < k:
        covered = set()
        for s in W: covered.update(s.nodes())
        uncovered = set(G.nodes()) - covered
        if not uncovered:
            if verbose: print(); print(f"Full coverage after {len(W)} hyperedges")
            break
        uncovered_list = list(uncovered)
        stats = {"subsets_checked":0,"density_calls":0,"distance_calls":0,"objective_calls":0}
        t0 = time.time()
        best_cand, best_new_cov, best_obj = None, 0, -1
        for _ in tqdm(range(trials), desc=f"  P1 hyper {len(W)+1} trials", leave=False):
            stats["subsets_checked"] += 1
            seed = random.choice(uncovered_list)
            all_nb = list(G.neighbors(seed)) + [seed]
            uncov_nb = [n for n in all_nb if n in uncovered]
            cov_nb = [n for n in all_nb if n not in uncovered]
            cov_sample = random.sample(cov_nb, min(cov_sample_size, len(cov_nb)))
            local = uncov_nb + cov_sample
            if len(local) > local_cap:
                u_in = [n for n in local if n in uncovered]
                c_in = [n for n in local if n not in uncovered]
                local = u_in + random.sample(c_in, min(cov_sample_size, len(c_in)))
            if len(local) < alpha: continue
            cand = DENSESTSUBGRAPH(G.subgraph(local).copy(), alpha, beta)
            if cand is None: continue
            new_cov = set(cand.nodes()) & uncovered
            if not new_cov: continue
            if not all(DISTANCE(cand, prev) != 0 for prev in W): continue
            obj = OBJECTIVE(W + [cand], lam)
            if (len(new_cov) > best_new_cov or
                (len(new_cov) == best_new_cov and obj > best_obj)):
                best_new_cov, best_obj, best_cand = len(new_cov), obj, cand
        t1 = time.time()
        if best_cand is None:
            if verbose:
                print()
                print(f"  Cannot find more coverage-expanding hyperedges.")
                print(f"  Remaining {len(uncovered)} nodes will go to fallback.")
            break
        W.append(best_cand)
        for v in best_cand.nodes(): membership[v] += 1
        if verbose:
            covered_now = len(covered) + best_new_cov
            print(f"=== Phase1 Hyperedge {len(W)} ===")
            print(f"  Size         : {best_cand.number_of_nodes()}")
            print(f"  Density      : {DENSITY(best_cand):.4f}")
            print(f"  New coverage : +{best_new_cov}  (total {covered_now}/{N_total} = {100*covered_now/N_total:.1f}%)")
            print(f"  Objective    : {best_obj:.4f}")
            print(f"  Time         : {t1 - t0:.2f}s")
            print(f"  Stats        : {stats}")
    p1 = len(W)

    if verbose:
        print()
        print("=" * 60)
        print("PHASE 2: Overlap enrichment")
        print("=" * 60)
    target = len(W) + phase2_extra
    while len(W) < target:
        stats = {"subsets_checked":0,"density_calls":0,"distance_calls":0,"objective_calls":0}
        t0 = time.time()
        best_cand, best_obj = None, -1
        # Phase 2 with per-user-membership boost: seed from the least-covered nodes in this pass.
        # Take the bottom 20% of nodes (by current membership count, restricted to those with neighbors)
        # and sample the seed from that pool each trial. This raises per-user membership for the long tail.
        nodes_with_nbrs = [n for n in G.nodes() if G.degree(n) > 0]
        nodes_with_nbrs.sort(key=lambda n: membership[n])
        low_mem_pool = nodes_with_nbrs[:max(1, len(nodes_with_nbrs)//5)]
        for _ in tqdm(range(trials), desc=f"  P2 hyper {len(W)+1} trials", leave=False):
            stats["subsets_checked"] += 1
            seed = random.choice(low_mem_pool)
            all_nb = list(G.neighbors(seed)) + [seed]
            local = ([seed] + random.sample(all_nb, local_cap-1)) if len(all_nb) > local_cap else all_nb
            if len(local) < alpha: continue
            cand = DENSESTSUBGRAPH(G.subgraph(local).copy(), alpha, beta)
            if cand is None: continue
            if not all(DISTANCE(cand, prev) != 0 for prev in W): continue
            obj = OBJECTIVE(W + [cand], lam)
            if obj > best_obj: best_obj, best_cand = obj, cand
        t1 = time.time()
        if best_cand is None:
            if verbose:
                print()
                print("  No more distinct subgraphs found.")
            break
        W.append(best_cand)
        for v in best_cand.nodes(): membership[v] += 1
        if verbose:
            print(f"=== Phase2 Hyperedge {len(W)} ===")
            print(f"  Size      : {best_cand.number_of_nodes()}")
            print(f"  Density   : {DENSITY(best_cand):.4f}")
            print(f"  Objective : {best_obj:.4f}")
            print(f"  Time      : {t1 - t0:.2f}s")
            print(f"  Stats     : {stats}")
    p2 = len(W) - p1

    # KNN-attachment fallback: for each uncovered node, attach to the existing hyperedge
    # whose members have the highest sum co-occurrence with this node.
    covered = set()
    for s in W: covered.update(s.nodes())
    uncov = set(G.nodes()) - covered
    n_attached = 0; n_singleton = 0
    if nbr_lookup is not None and uncov:
        for node in tqdm(uncov, desc="  KNN-attach fallback"):
            nbr_set = {v: c for v, c in nbr_lookup[node]}
            best_idx, best_score = -1, 0
            for idx, sg in enumerate(W):
                score = sum(nbr_set.get(m, 0) for m in sg.nodes())
                if score > best_score:
                    best_score = score; best_idx = idx
            if best_idx >= 0:
                W[best_idx].add_node(node)
                for m, c in nbr_lookup[node]:
                    if m in W[best_idx]: W[best_idx].add_edge(node, m)
                n_attached += 1
            else:
                sg = nx.Graph(); sg.add_node(node); W.append(sg); n_singleton += 1
    else:
        for node in uncov:
            sg = nx.Graph(); sg.add_node(node); W.append(sg); n_singleton += 1

    if verbose:
        print()
        print("=" * 60)
        print("DOSAGE_TWO_PHASE COMPLETE")
        print("=" * 60)
        print(f"  Phase 1 hyperedges : {p1}")
        print(f"  Phase 2 hyperedges : {p2}")
        print(f"  Attached fallback  : {n_attached}  (uncovered nodes attached to nearest hyperedge)")
        print(f"  Singleton fallback : {n_singleton}  (no usable neighbor — last resort)")
        print(f"  Total hyperedges   : {len(W)}")
        covered_final = set()
        for s in W: covered_final.update(s.nodes())
        print(f"  Final coverage     : {len(covered_final)}/{N_total} = {100*len(covered_final)/N_total:.1f}%")
        dosage_covered = set()
        for s in W[:p1+p2]: dosage_covered.update(s.nodes())
        print(f"  DOSAGE-only cov    : {len(dosage_covered)}/{N_total} = {100*len(dosage_covered)/N_total:.1f}%  (before any fallback)")
        print(f"  Total time         : {time.time()-t_overall:.2f}s")
        # Membership distribution across all hyperedges (including KNN-attached and singletons)
        final_mem = {n: 0 for n in G.nodes()}
        for sg in W:
            for v in sg.nodes(): final_mem[v] += 1
        mem_vals = list(final_mem.values())
        import numpy as _np
        print(f"  Membership stats   : mean={_np.mean(mem_vals):.2f}  median={_np.median(mem_vals):.0f}  min={min(mem_vals)}  max={max(mem_vals)}  bottom10%={_np.percentile(mem_vals, 10):.0f}")
    return W


In [8]:
DOSAGE_PARAMS = dict(k=600, lam=2, alpha=20, beta=80, trials=100, phase2_extra=200)

print('===== DOSAGE on user-user graph =====')
random.seed(SEED); np.random.seed(SEED)
t0 = time.time()
W_user = DOSAGE_TWO_PHASE(G_user, **DOSAGE_PARAMS, nbr_lookup=uu_nbrs)
print(f'  Total time: {time.time()-t0:.1f}s\n')

print('===== DOSAGE on item-item graph =====')
random.seed(SEED); np.random.seed(SEED)
t0 = time.time()
W_item = DOSAGE_TWO_PHASE(G_item, **DOSAGE_PARAMS, nbr_lookup=ii_nbrs)
print(f'  Total time: {time.time()-t0:.1f}s')

===== DOSAGE on user-user graph =====
PHASE 1: Coverage-first


=== Phase1 Hyperedge 1 ===
  Size         : 80
  Density      : 39.1125
  New coverage : +80  (total 80/897 = 8.9%)
  Objective    : 39.1125
  Time         : 32.95s
  Stats        : {'subsets_checked': 100, 'density_calls': 0, 'distance_calls': 0, 'objective_calls': 0}


=== Phase1 Hyperedge 2 ===
  Size         : 80
  Density      : 33.0250
  New coverage : +72  (total 152/897 = 16.9%)
  Objective    : 76.1175
  Time         : 21.36s
  Stats        : {'subsets_checked': 100, 'density_calls': 0, 'distance_calls': 0, 'objective_calls': 0}


=== Phase1 Hyperedge 3 ===
  Size         : 80
  Density      : 36.5000
  New coverage : +70  (total 222/897 = 24.7%)
  Objective    : 120.5909
  Time         : 17.15s
  Stats        : {'subsets_checked': 100, 'density_calls': 0, 'distance_calls': 0, 'objective_calls': 0}


=== Phase1 Hyperedge 4 ===
  Size         : 79
  Density      : 36.3165
  New coverage : +67  (total 289/897 = 32.2%)
  Objective    : 168.8808
  Time         : 9.62s
  Stats        : {'subsets_checked': 100, 'density_calls': 0, 'distance_calls': 0, 'objective_calls': 0}


...

=== Phase2 Hyperedge 216 ===
  Size      : 24
  Density   : 11.5000
  Objective : 96356.7857
  Time      : 10.16s
  Stats     : {'subsets_checked': 100, 'density_calls': 0, 'distance_calls': 0, 'objective_calls': 0}


=== Phase2 Hyperedge 217 ===
  Size      : 29
  Density   : 13.9655
  Objective : 97212.8254
  Time      : 8.84s
  Stats     : {'subsets_checked': 100, 'density_calls': 0, 'distance_calls': 0, 'objective_calls': 0}


=== Phase2 Hyperedge 218 ===
  Size      : 55
  Density   : 26.7636
  Objective : 98072.3484
  Time      : 9.93s
  Stats     : {'subsets_checked': 100, 'density_calls': 0, 'distance_calls': 0, 'objective_calls': 0}


  KNN-attach fallback: 100%|██████████| 123/123 [00:00<00:00, 713.21it/s]


DOSAGE_TWO_PHASE COMPLETE
  Phase 1 hyperedges : 18
  Phase 2 hyperedges : 200
  Attached fallback  : 123  (uncovered nodes attached to nearest hyperedge)
  Singleton fallback : 0  (no usable neighbor — last resort)
  Total hyperedges   : 218
  Final coverage     : 824/824 = 100.0%
  DOSAGE-only cov    : 824/824 = 100.0%  (before any fallback)
  Total time         : 1228.50s
  Membership stats   : mean=17.43  median=5  min=1  max=149  bottom10%=1
  Total time: 1228.5s


In [9]:
def hyperedges_to_sparse(W, n_nodes):
    """Convert list of NetworkX subgraphs (hyperedges) to a sparse (n_nodes × num_hyperedges) matrix."""
    rows, cols = [], []
    for j, sg in enumerate(W):
        for u in sg.nodes():
            rows.append(u); cols.append(j)
    return sp.coo_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)),
                          shape=(n_nodes, len(W))).tocsr()

H_ui_dosage_sp = hyperedges_to_sparse(W_user, n_users)   # user-vertex hypergraph from DOSAGE
H_iu_dosage_sp = hyperedges_to_sparse(W_item, n_items)   # item-vertex hypergraph from DOSAGE
print(f'DOSAGE H_ui: shape={H_ui_dosage_sp.shape}, nnz={H_ui_dosage_sp.nnz}')
print(f'DOSAGE H_iu: shape={H_iu_dosage_sp.shape}, nnz={H_iu_dosage_sp.nnz}')

DOSAGE H_ui: shape=(897, 224), nnz=16919
DOSAGE H_iu: shape=(824, 218), nnz=14361


## 4. The HGNN smoothing operator (used by DHCF)

DHCF's `smoothing_with_HGNN(X)` is the standard normalized hypergraph propagation:

$$\hat X = D_v^{-1/2}\, H\, W_e\, D_e^{-1}\, H^T\, D_v^{-1/2}\, X$$

where `D_v`, `D_e` are vertex / hyperedge degree diagonals and `W_e=I` (uniform hyperedge weights). We precompute the operator `S = D_v^{-1/2} H D_e^{-1} H^T D_v^{-1/2}` once per hypergraph and apply it as `X ← S @ X` in the forward pass.

In [10]:
def hgnn_propagation_op(H_sp):
    """Build the sparse propagation matrix S = Dv^-1/2 H De^-1 H^T Dv^-1/2 from an incidence matrix."""
    H = H_sp.tocsr()
    dv = np.asarray(H.sum(axis=1)).flatten()                 # vertex degrees
    de = np.asarray(H.sum(axis=0)).flatten()                 # hyperedge degrees
    dv_inv_sqrt = np.where(dv > 0, np.power(dv, -0.5), 0)
    de_inv     = np.where(de > 0, 1.0 / de, 0)
    Dv_is = sp.diags(dv_inv_sqrt)
    De_i  = sp.diags(de_inv)
    S = Dv_is @ H @ De_i @ H.T @ Dv_is
    S = S.tocoo()
    idx = torch.from_numpy(np.vstack([S.row, S.col])).long()
    val = torch.from_numpy(S.data.astype(np.float32))
    return torch.sparse_coo_tensor(idx, val, S.shape).coalesce()

print('Building propagation operators...')
S_ui_orig   = hgnn_propagation_op(H_ui_orig_sp).to(device)     # for user embeddings, DHCF original
S_iu_orig   = hgnn_propagation_op(H_iu_orig_sp).to(device)     # for item embeddings, DHCF original
S_ui_dosage = hgnn_propagation_op(H_ui_dosage_sp).to(device)   # for user embeddings, DOSAGE
S_iu_dosage = hgnn_propagation_op(H_iu_dosage_sp).to(device)   # for item embeddings, DOSAGE
print(f'S_ui_orig: {tuple(S_ui_orig.shape)}, nnz={S_ui_orig._nnz()}')
print(f'S_iu_orig: {tuple(S_iu_orig.shape)}, nnz={S_iu_orig._nnz()}')
print(f'S_ui_dosage: {tuple(S_ui_dosage.shape)}, nnz={S_ui_dosage._nnz()}')
print(f'S_iu_dosage: {tuple(S_iu_dosage.shape)}, nnz={S_iu_dosage._nnz()}')

Building propagation operators...
S_ui_orig: (897, 897), nnz=671927
S_iu_orig: (824, 824), nnz=530010
S_ui_dosage: (897, 897), nnz=294887
S_iu_dosage: (824, 824), nnz=207102


## 5. DHCF model — ported verbatim from `dhg.models.hypergraphs.dhcf`

Single class. The two variants differ only in which `(S_ui, S_iu)` pair is passed in.

In [11]:
class DHCF(nn.Module):
    """Verbatim port of dhg.models.hypergraphs.DHCF.
    
    The original takes Hypergraph objects and calls hg.smoothing_with_HGNN(x).
    Here we precomputed the smoothing operators S_ui and S_iu as sparse tensors and just spmm.
    Functionally identical.
    """
    def __init__(self, num_users, num_items, emb_dim=64, num_layers=3, drop_rate=0.5):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.num_layers = num_layers
        self.drop_rate = drop_rate
        self.u_embedding = nn.Embedding(num_users, emb_dim)
        self.i_embedding = nn.Embedding(num_items, emb_dim)
        self.W_gc = nn.ModuleList([nn.Linear(emb_dim, emb_dim) for _ in range(num_layers)])
        self.W_bi = nn.ModuleList([nn.Linear(emb_dim, emb_dim) for _ in range(num_layers)])
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.u_embedding.weight)
        nn.init.xavier_uniform_(self.i_embedding.weight)
        for w_gc, w_bi in zip(self.W_gc, self.W_bi):
            nn.init.xavier_uniform_(w_gc.weight); nn.init.constant_(w_gc.bias, 0)
            nn.init.xavier_uniform_(w_bi.weight); nn.init.constant_(w_bi.bias, 0)

    def forward(self, S_ui, S_iu):
        u_embs = self.u_embedding.weight
        i_embs = self.i_embedding.weight
        all_embs = torch.cat([u_embs, i_embs], dim=0)
        embs_list = [all_embs]
        for idx in range(self.num_layers):
            u_e, i_e = torch.split(all_embs, [self.num_users, self.num_items], dim=0)
            # Two JHConv layers — one per side
            u_e = torch.sparse.mm(S_ui, u_e)
            i_e = torch.sparse.mm(S_iu, i_e)
            g_embs = torch.cat([u_e, i_e], dim=0)
            sum_embs = F.leaky_relu(self.W_gc[idx](g_embs) + all_embs, negative_slope=0.2)
            bi_embs = all_embs * g_embs
            bi_embs = F.leaky_relu(self.W_bi[idx](bi_embs), negative_slope=0.2)
            all_embs = sum_embs + bi_embs
            all_embs = F.dropout(all_embs, p=self.drop_rate, training=self.training)
            all_embs = F.normalize(all_embs, p=2, dim=1)
            embs_list.append(all_embs)
        embs = torch.stack(embs_list, dim=1).mean(dim=1)
        return torch.split(embs, [self.num_users, self.num_items], dim=0)

## 6. Training (BPR loss) and full-ranking evaluation

In [12]:
train_users = np.array([u for u,_ in train_pairs])
train_items = np.array([i for _,i in train_pairs])
user_pos_arr = [np.array(sorted(train_user_pos[u])) for u in range(n_users)]

def sample_neg_all():
    """Sample one negative per training pair (all ~460k at once)."""
    us = train_users; ps = train_items
    ns = np.random.randint(0, n_items, size=len(us))
    for k in range(len(us)):
        while ns[k] in train_user_pos[us[k]]:
            ns[k] = np.random.randint(0, n_items)
    return us, ps, ns

@torch.no_grad()
def evaluate(model, S_ui, S_iu, K=20, batch=512):
    model.eval()
    uE, iE = model(S_ui, S_iu)
    test_users = list(test_by_user.keys())
    recalls, ndcgs = [], []
    log2 = np.log2(np.arange(2, K+2))
    for s in range(0, len(test_users), batch):
        bu_idx = test_users[s:s+batch]
        bu = uE[bu_idx]
        scores = bu @ iE.t()
        for k, u in enumerate(bu_idx):
            if len(user_pos_arr[u]) > 0:
                scores[k, user_pos_arr[u]] = -1e9
        topk = torch.topk(scores, K, dim=-1).indices.cpu().numpy()
        for k, u in enumerate(bu_idx):
            gt = test_by_user[u]
            hits = np.array([1.0 if topk[k,j] in gt else 0.0 for j in range(K)])
            recalls.append(hits.sum() / len(gt))
            dcg = (hits / log2).sum()
            ideal = (1.0 / log2[:min(len(gt), K)]).sum()
            ndcgs.append(dcg / ideal if ideal > 0 else 0.0)
    return float(np.mean(recalls)), float(np.mean(ndcgs))

def train_dhcf(model, S_ui, S_iu, epochs=60, lr=1e-3, weight_decay=1e-4, eval_every=3, tag=""):
    """One forward + one backward per epoch on the full train set.
    Previously we did a forward per inner batch (~100/epoch) which was the bottleneck."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_recall, best_ndcg = 0.0, 0.0
    for ep in range(1, epochs+1):
        model.train(); t0 = time.time()
        us, ps, ns = sample_neg_all()
        us_t = torch.from_numpy(us).long().to(device)
        ps_t = torch.from_numpy(ps).long().to(device)
        ns_t = torch.from_numpy(ns).long().to(device)
        uE, iE = model(S_ui, S_iu)
        pos = (uE[us_t] * iE[ps_t]).sum(-1)
        neg = (uE[us_t] * iE[ns_t]).sum(-1)
        bpr = -F.logsigmoid(pos - neg).mean()
        l2 = (model.u_embedding.weight[us_t].pow(2).mean() +
              model.i_embedding.weight[ps_t].pow(2).mean() +
              model.i_embedding.weight[ns_t].pow(2).mean())
        loss = bpr + weight_decay * l2
        opt.zero_grad(); loss.backward(); opt.step()
        msg = f"[{tag}] Ep {ep:02d}  loss={loss.item():.4f}  ({time.time()-t0:.1f}s)"
        if ep % eval_every == 0 or ep == epochs:
            r, n = evaluate(model, S_ui, S_iu)
            msg += f"   Recall@20={r:.4f}  NDCG@20={n:.4f}"
            if r > best_recall: best_recall, best_ndcg = r, n
        print(msg)
    return best_recall, best_ndcg


## 7. Run both variants

In [13]:
EPOCHS = 60
EMB_DIM = 32
NUM_LAYERS = 2

print("="*70); print("Variant A: DHCF (original -- item/user-derived hyperedges)"); print("="*70)
torch.manual_seed(SEED)
m_orig = DHCF(n_users, n_items, emb_dim=EMB_DIM, num_layers=NUM_LAYERS).to(device)
rA, nA = train_dhcf(m_orig, S_ui_orig, S_iu_orig, epochs=EPOCHS, tag="DHCF")


Variant A: DHCF (original -- item/user-derived hyperedges)
[DHCF] Ep 01  loss=0.6933  (1.2s)
[DHCF] Ep 02  loss=0.6928  (0.1s)
[DHCF] Ep 03  loss=0.6916  (0.1s)   Recall@20=0.0269  NDCG@20=0.0237
[DHCF] Ep 04  loss=0.6909  (0.1s)
[DHCF] Ep 05  loss=0.6903  (0.1s)
[DHCF] Ep 06  loss=0.6897  (0.1s)   Recall@20=0.0321  NDCG@20=0.0276
[DHCF] Ep 07  loss=0.6891  (0.1s)
[DHCF] Ep 08  loss=0.6886  (0.1s)
[DHCF] Ep 09  loss=0.6886  (0.1s)   Recall@20=0.0407  NDCG@20=0.0333
[DHCF] Ep 10  loss=0.6874  (0.1s)
[DHCF] Ep 11  loss=0.6869  (0.1s)
[DHCF] Ep 12  loss=0.6862  (0.1s)   Recall@20=0.0442  NDCG@20=0.0376
[DHCF] Ep 13  loss=0.6859  (0.1s)
[DHCF] Ep 14  loss=0.6856  (0.1s)
[DHCF] Ep 15  loss=0.6852  (0.1s)   Recall@20=0.0521  NDCG@20=0.0438
[DHCF] Ep 16  loss=0.6841  (0.1s)
[DHCF] Ep 17  loss=0.6836  (0.1s)
[DHCF] Ep 18  loss=0.6838  (0.1s)   Recall@20=0.0595  NDCG@20=0.0492
[DHCF] Ep 19  loss=0.6823  (0.1s)
[DHCF] Ep 20  loss=0.6818  (0.1s)
[DHCF] Ep 21  loss=0.6805  (0.1s)   Recall@20=0.066

In [14]:
print("="*70); print("Variant B: DOSAGE-DHCF (DOSAGE-derived hyperedges)"); print("="*70)
torch.manual_seed(SEED)
m_dos = DHCF(n_users, n_items, emb_dim=EMB_DIM, num_layers=NUM_LAYERS).to(device)
rB, nB = train_dhcf(m_dos, S_ui_dosage, S_iu_dosage, epochs=EPOCHS, tag="DOSAGE-DHCF")


Variant B: DOSAGE-DHCF (DOSAGE-derived hyperedges)
[DOSAGE-DHCF] Ep 01  loss=0.6930  (0.0s)
[DOSAGE-DHCF] Ep 02  loss=0.6922  (0.1s)
[DOSAGE-DHCF] Ep 03  loss=0.6918  (0.0s)   Recall@20=0.0285  NDCG@20=0.0240
[DOSAGE-DHCF] Ep 04  loss=0.6907  (0.0s)
[DOSAGE-DHCF] Ep 05  loss=0.6898  (0.0s)
[DOSAGE-DHCF] Ep 06  loss=0.6888  (0.0s)   Recall@20=0.0339  NDCG@20=0.0287
[DOSAGE-DHCF] Ep 07  loss=0.6882  (0.1s)
[DOSAGE-DHCF] Ep 08  loss=0.6878  (0.1s)
[DOSAGE-DHCF] Ep 09  loss=0.6870  (0.0s)   Recall@20=0.0391  NDCG@20=0.0329
[DOSAGE-DHCF] Ep 10  loss=0.6860  (0.0s)
[DOSAGE-DHCF] Ep 11  loss=0.6861  (0.0s)
[DOSAGE-DHCF] Ep 12  loss=0.6846  (0.0s)   Recall@20=0.0448  NDCG@20=0.0379
[DOSAGE-DHCF] Ep 13  loss=0.6846  (0.0s)
[DOSAGE-DHCF] Ep 14  loss=0.6845  (0.0s)
[DOSAGE-DHCF] Ep 15  loss=0.6836  (0.0s)   Recall@20=0.0529  NDCG@20=0.0448
[DOSAGE-DHCF] Ep 16  loss=0.6830  (0.0s)
[DOSAGE-DHCF] Ep 17  loss=0.6817  (0.0s)
[DOSAGE-DHCF] Ep 18  loss=0.6818  (0.0s)   Recall@20=0.0573  NDCG@20=0.0500
[

In [15]:
print('\n' + '='*60)
print(f'{"Model":<25} {"Recall@20":>12} {"NDCG@20":>12}')
print('-'*60)
print(f'{"DHCF (item-derived)":<25} {rA:>12.4f} {nA:>12.4f}')
print(f'{"DOSAGE-DHCF":<25} {rB:>12.4f} {nB:>12.4f}')
print('-'*60)
print(f'{"Δ (DOSAGE - orig)":<25} {rB-rA:>+12.4f} {nB-nA:>+12.4f}')
if rA > 0:
    print(f'{"Relative":<25} {(rB-rA)/rA*100:>+11.2f}% {(nB-nA)/nA*100:>+11.2f}%')
print('='*60)


Model                        Recall@20      NDCG@20
------------------------------------------------------------
DHCF (item-derived)             0.1175       0.1027
DOSAGE-DHCF                     0.1022       0.0927
------------------------------------------------------------
Δ (DOSAGE - orig)              -0.0153      -0.0100
Relative                       -13.04%       -9.73%


## Notes

**What this comparison isolates.** Same DHCF backbone, same data, same training/eval. The *only* difference is the hyperedge construction:

| | DHCF original | DOSAGE-DHCF |
|---|---|---|
| User-vertex hyperedges | one per item = its users | DOSAGE on user-user co-occurrence |
| Item-vertex hyperedges | one per user = their items | DOSAGE on item-item co-occurrence |
| #user-hyperedges | n_items (~3000) | phase1 + phase2 + singletons (~250+) |
| #item-hyperedges | n_users (~6000) | phase1 + phase2 + singletons (~250+) |
| Construction | trivial, dense | DOSAGE, structural |

**Why this is a fair fight.** Both are **fixed**, both are **derived from the interaction matrix**, neither involves learning the hyperedges. The model sees the hyperedges through the same propagation operator `S = Dv^-1/2 H De^-1 H^T Dv^-1/2`. So any performance delta is genuinely attributable to the construction choice.

**What to watch for.** DHCF original has *much more* hyperedges (~3000 vs ~250). If DHCF original wins, it might be because of sheer hyperedge count rather than construction quality. To control for this, try increasing DOSAGE's `k` to 1000+ or run a follow-up where DHCF original is restricted to a random subset of 250 hyperedges.

**Knobs to tune:** `MIN_SHARED_USER`, `MIN_SHARED_ITEM` (collab graph density), DOSAGE `k`/`alpha`/`beta`/`trials`, model `EMB_DIM`/`NUM_LAYERS`, `EPOCHS`.